In [ ]:
# %pip install torch
# %pip install transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 62.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
# %pip install wget

In [ ]:
# !wget https://raw.githubusercontent.com/microsoft/CodeBERT/master/UniXcoder/unixcoder.py

In [1]:
# Testing torch
import torch

print("Number of GPU: ", torch.cuda.device_count())
print("GPU Name: ", torch.cuda.get_device_name())


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Number of GPU:  1
GPU Name:  NVIDIA GeForce RTX 4050 Laptop GPU
Using device: cuda


Import UniXCoder and torch
  - Test if CUDA is installed in the device, if not, it will use CPU instead

In [ ]:
from unixcoder import UniXcoder
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UniXcoder("microsoft/unixcoder-base-nine")
model.to(device)

Encoder-Only mode
  1. Code and NL Embeddings

In [ ]:
# Encode maximum function
func = "def f(a,b): return a if a > b else return b"
tokens_ids = model.tokenize([func], max_length=512, mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings, max_func_embeddings = model(source_ids)

# Encode minimum function
func = "def f(a,b): return a if a < b else return b"
tokens_ids = model.tokenize([func], max_length=512, mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings, min_func_embeddings = model(source_ids)

# Encode natural language
nl = "return maximum value"
tokens_ids = model.tokenize([nl], max_length=512, mode="<encoder-only>")
source_ids = torch.tensor(tokens_ids).to(device)
tokens_embeddings, nl_embeddings = model(source_ids)

print(max_func_embeddings.shape)
print(max_func_embeddings)


2. Similarity between code and NL
  - Normalize two embeddings (Min and Max)
  - Calculate the cosine similarity between two embedded vectors (Dot Product)

In [ ]:
norm_max_func_embeddings = torch.nn.functional.normalize(max_func_embeddings, p=2, dim=1)
norm_min_func_embeddings = torch.nn.functional.normalize(min_func_embeddings, p=2, dim=1)
norm_nl_embedding = torch.nn.functional.normalize(nl_embeddings, p=2, dim=1)

# Calculate the similarity
max_func_nl_similarity = torch.einsum("ac,bc->ab",norm_max_func_embeddings, norm_nl_embedding)
min_func_nl_similarity = torch.einsum("ac,bc->ab", norm_min_func_embeddings, norm_nl_embedding)

print(max_func_nl_similarity)
print(min_func_nl_similarity)


## Code Search Tasks
1. NL to Code (Code Completion)
2. Code to Text (Code Summarization)

In [ ]:
# NL to Code
# def extract_code(out):
#   lines = out.split("\n")
#   lines = [line for line in lines if not line.strip().startswith("@")]
#   return "\n".join(lines)


input = """write a function that returns a maximum value"""
input_tokens_ids = model.tokenize([input], max_length=512, mode="<decoder-only>")
input_source_id = torch.tensor(input_tokens_ids).to(device)
pred_id = model.generate(input_source_id, beam_size=3, decoder_only=True, max_length=128)
pred = model.decode(pred_id)
print(pred[0][0])

